# AION Market Sentiment Analysis

**Real-time sentiment intelligence for Indian financial markets**

- 98.55% accuracy on Indian financial news
- <100ms inference latency
- 592 NSE tickers covered

## Quick Start

In [ ]:
# Install packages (run once)
# !pip install aion-sentiment aion-sectormap aion-volweight pandas

In [ ]:
import pandas as pd
from aion_sentiment import AIONSentimentAnalyzer
from aion_sectormap import SectorMapper
from aion_volweight import weight_confidence, get_regime

# Initialize
analyzer = AIONSentimentAnalyzer()
mapper = SectorMapper()

## 1. Single Headline Analysis

In [ ]:
# Analyze single headline
headline = "Reliance Industries reports record quarterly profits"
result = analyzer.predict(headline)

print(f"Headline: {headline}")
print(f"Sentiment: {result[0]['label']}")
print(f"Confidence: {result[0]['confidence']:.2%}")

## 2. Batch Analysis

In [ ]:
# Analyze multiple headlines
headlines = [
    "Stock market reaches all-time high on optimism",
    "Market crashes on recession fears",
    "Trading volume remains below average",
    "Tech stocks surge as earnings beat",
    "Investors panic as banks collapse"
]

results = analyzer.predict(headlines)

df = pd.DataFrame({
    'Headline': headlines,
    'Sentiment': [r['label'] for r in results],
    'Confidence': [f"{r['confidence']:.1%}" for r in results]
})

df

## 3. Sector Mapping

In [ ]:
# Map tickers to sectors
df_tickers = pd.DataFrame({
    'ticker': ['RELIANCE', 'TCS', 'HDFCBANK', 'TATAMOTORS', 'ITC'],
    'headline': [
        'Record profits reported',
        'Major deal win announced',
        'Rural expansion planned',
        'Supply chain challenges',
        'New FMCG segment launch'
    ]
})

# Map sectors
df_tickers = mapper.map(df_tickers, ticker_column='ticker')

# Analyze sentiment
df_tickers = analyzer.analyze(df_tickers, text_column='headline')

df_tickers[['ticker', 'sector', 'headline', 'sentiment_label', 'sentiment_confidence']]

## 4. VIX-Adjusted Confidence

In [ ]:
# Adjust confidence based on VIX regime
vix_values = [10, 14, 18, 22, 28]

print("VIX Regime Reference:")
print("-" * 50)
for vix in vix_values:
    regime = get_regime(vix)
    print(f"VIX={vix:2.0f} → {regime:6} regime")

# Apply VIX adjustment
current_vix = 18  # HIGH regime
df_adjusted = weight_confidence(df_tickers, vix_value=current_vix)

print(f"\nVIX={current_vix} (HIGH regime - 20% confidence discount)")
df_adjusted[['ticker', 'sentiment_confidence', 'sentiment_confidence_adjusted']]

## 5. Load Sample Dataset

In [ ]:
# Load sample data
sample_df = pd.read_csv('data/news_sentiment_sample.csv')
sample_df['publish_date'] = pd.to_datetime(sample_df['publish_date'])

print(f"Loaded {len(sample_df)} samples")
sample_df.head()

## 6. Sentiment Distribution

In [ ]:
import matplotlib.pyplot as plt

# Sentiment distribution
sentiment_counts = sample_df['sentiment_label'].value_counts()

plt.figure(figsize=(10, 6))
plt.bar(sentiment_counts.index, sentiment_counts.values, color=['green', 'gray', 'red'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment Label')
plt.ylabel('Count')
plt.show()

print(sentiment_counts)

## 7. Sector Sentiment Heatmap

In [ ]:
# Aggregate by sector
sector_sentiment = sample_df.groupby('sector').agg({
    'sentiment_label': lambda x: (x == 'positive').mean() * 100,
    'confidence_score': 'mean'
}).reset_index()

sector_sentiment.columns = ['Sector', 'Bullish %', 'Avg Confidence']
sector_sentiment = sector_sentiment.sort_values('Bullish %', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(sector_sentiment['Sector'], sector_sentiment['Bullish %'], color='green', alpha=0.7)
plt.title('Sector Sentiment Heatmap')
plt.xlabel('Bullish %')
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

sector_sentiment

## 8. Performance Benchmarks

In [ ]:
import time

# Benchmark single prediction
start = time.time()
for _ in range(100):
    analyzer.predict("Market reaches all-time high")
single_latency = (time.time() - start) / 100 * 1000

print(f"Single prediction latency: {single_latency:.1f}ms")
print(f"Throughput: {1000/single_latency:.0f} predictions/sec")

# Benchmark batch prediction
batch_size = 100
headlines = ["Market reaches all-time high"] * batch_size

start = time.time()
analyzer.predict(headlines)
batch_latency = (time.time() - start) * 1000

print(f"\nBatch prediction ({batch_size} items): {batch_latency:.1f}ms")
print(f"Throughput: {batch_size/(batch_latency/1000):.0f} predictions/sec")

## 9. Real-World Use Case: Trading Signal

In [ ]:
def generate_trading_signal(headline, vix_value=18):
    """
    Generate trading signal based on sentiment and VIX regime.
    
    Returns: BUY, SELL, or HOLD
    """
    # Analyze sentiment
    result = analyzer.predict(headline)[0]
    sentiment = result['label']
    confidence = result['confidence']
    
    # Adjust for VIX
    regime = get_regime(vix_value)
    if regime == 'PANIC':
        confidence *= 0.5
    elif regime == 'HIGH':
        confidence *= 0.8
    
    # Generate signal
    if sentiment == 'positive' and confidence > 0.85:
        return 'BUY', confidence
    elif sentiment == 'negative' and confidence > 0.85:
        return 'SELL', confidence
    else:
        return 'HOLD', confidence

# Test signals
test_headlines = [
    "Reliance reports record profits",
    "Market crashes on recession fears",
    "Trading volume remains average"
]

print("Trading Signals (VIX=18):")
print("-" * 60)
for headline in test_headlines:
    signal, conf = generate_trading_signal(headline, vix_value=18)
    print(f"{headline:40} → {signal:4} (confidence: {conf:.1%})")

## Next Steps

- Explore [GitHub](https://github.com/AION-Analytics/market-sentiments) for more examples
- Read [documentation](https://github.com/AION-Analytics/market-sentiments/blob/main/README.md)
- Try [aion-newsimpact](https://github.com/AION-Analytics/market-sentiments/tree/main/aion-newsimpact) for historical impact analysis
- Join [Discussions](https://github.com/AION-Analytics/market-sentiments/discussions)